# Module 5 — Delta Live Tables (DLT)
Exam domain: **Data Processing**

Runs standalone in Google Colab. **Note:** DLT (`import dlt`, `@dlt.table`,
`@dlt.expect`) is a Databricks-managed pipeline runtime — the `dlt` module only
exists inside a running DLT pipeline and cannot be `pip install`ed or executed
in Colab. This notebook shows (a) the real DLT syntax as reference code you
would put in a `.py` file for a DLT pipeline, and (b) a runnable "simulation"
using plain PySpark functions that mirror the same declarative logic, so you can
still exercise the concepts here.

In [ ]:
!pip install -q pyspark==3.5.1 delta-spark==3.2.0

In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

builder = (SparkSession.builder
    .appName("Module5-DLT-Simulation")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog"))

spark = configure_spark_with_delta_pip(builder).getOrCreate()

## Reference: real DLT pipeline code
This is what you would actually write for Databricks DLT — copy it into a `.py`
source file and attach it to a DLT pipeline in the workspace; it will not run
as a notebook cell.

In [ ]:
# --- reference only: real DLT syntax, requires a DLT pipeline to execute ---
# import dlt
# from pyspark.sql import functions as F
#
# @dlt.table(comment="Raw events, ingested as-is")
# def bronze_events():
#     return spark.readStream.format("cloudFiles").option("cloudFiles.format", "json").load("/landing/events")
#
# @dlt.table(comment="Cleaned, validated events")
# @dlt.expect_or_drop("valid_amount", "amount IS NOT NULL AND amount > 0")
# @dlt.expect("valid_name", "name IS NOT NULL")
# def silver_events():
#     return dlt.read_stream("bronze_events").withColumn("amount", F.col("amount_raw").cast("double"))
#
# @dlt.table(comment="Daily aggregates for BI")
# def gold_daily_summary():
#     return dlt.read("silver_events").groupBy("event_date").agg(F.sum("amount").alias("total_amount"))


## Simulation with plain PySpark
Same three-layer logic and the same idea as `@dlt.expect_or_drop` (a data
quality rule that silently drops violating rows while the pipeline keeps
running) — reimplemented as ordinary functions so it's runnable here.

In [ ]:
raw = spark.createDataFrame([
    (1, "Alice", "2024-01-01", "100.5"),
    (2, "Bob", "2024-01-02", "bad_value"),
    (3, None, "2024-01-03", "-5.0"),
], ["id", "name", "event_date", "amount_raw"])

def bronze_events(df):
    return df.withColumn("ingestion_ts", F.current_timestamp())

def expect_or_drop(df, condition):
    """Simulates @dlt.expect_or_drop: rows failing `condition` are dropped, not fatal."""
    return df.filter(condition)

def silver_events(df):
    df = df.withColumn("amount", F.col("amount_raw").cast("double"))
    df = expect_or_drop(df, "amount IS NOT NULL AND amount > 0")
    df = expect_or_drop(df, "name IS NOT NULL")
    return df.withColumn("event_date", F.to_date("event_date")).drop("amount_raw")

def gold_daily_summary(df):
    return df.groupBy("event_date").agg(F.sum("amount").alias("total_amount"))

bronze_df = bronze_events(raw)
silver_df = silver_events(bronze_df)
gold_df = gold_daily_summary(silver_df)

silver_df.show()
gold_df.show()

## Key DLT concepts to know for the exam
- **Declarative**: you describe *what* each table is, DLT figures out execution
  order from dependencies (`dlt.read` / `dlt.read_stream` calls).
- **Expectations**: `@dlt.expect` (warn, keep row), `@dlt.expect_or_drop` (drop
  row), `@dlt.expect_or_fail` (stop the pipeline).
- **Triggered vs. continuous** pipeline modes, similar to the streaming triggers
  in Module 4.
- **Live tables vs. streaming tables**: streaming tables process each input row
  once (incremental); live/materialized views recompute fully.